In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact

# Load the ADPDF.npz file
datao = np.load("data/original.npz")

print(datao['q'].shape)


In [ ]:
from ipywidgets import interact

def plot_field(atom):
    plt.figure()
    plt.plot(r, G_center[atom], label=f"G_center {atom}")
    plt.plot()
    plt.legend()
    plt.show()

interact(plot_field, atom=elements)


In [ ]:
#!/usr/bin/env python
import numpy as np
from diffpy.srreal.pdfcalculator import PDFCalculator, fftgtof, fftftog
from diffpy.srreal.scatteringfactortable import SFTXray, SFTNeutron
from diffpy.structure import loadStructure

# ---- helpers ----
def lorch_window(Q, Qmax):
    # standard Lorch modification to suppress termination ripples
    x = np.pi * Q / Qmax
    out = np.ones_like(Q)
    nz = Q > 0
    out[nz] = np.sin(x[nz]) / x[nz]
    return out

def compute_pair_partials_F(stru, pc):
    # total -> defines r-grid and Q-grid via FFT
    r, Gtot = pc(stru)
    n_r = Gtot.size
    dr = r[1] - r[0]
    Ftot, dq = fftgtof(Gtot, dr)
    Q = np.arange(Ftot.size) * dq

    elems = sorted({a.element for a in stru})
    Fpair = {}
    for i, A in enumerate(elems):
        for B in elems[i:]:
            pc.maskAllPairs(False)         # all OFF
            pc.setTypeMask(A, B, True)     # just this unordered pair
            _, Gab = pc(stru)
            Fab, dq2 = fftgtof(Gab, dr)
            if abs(dq2 - dq) > 1e-12:
                raise RuntimeError("dq mismatch; fix r-grid for all calls.")
            Fpair[(A, B)] = Fab
    return elems, r, n_r, dr, Gtot, Ftot, Q, dq, Fpair

def element_centered_dpdfs_Qspace(
    cif_path,
    rmax=60.0, rstep=0.01, qmin=0.1, qmax=30.0,
    scattering="X", use_constant_weights=True, use_lorch=True
):
    stru = loadStructure(cif_path)

    # PDFCalculator on fixed r-grid; rmin MUST be 0 for clean FFT pairing
    pc = PDFCalculator()
    pc.rmin = 0.0
    pc.rmax = float(rmax)
    pc.rstep = float(rstep)
    pc.qmin = float(qmin)
    pc.qmax = float(qmax)

    # scattering table
    sft = SFTXray() if (isinstance(scattering, str) and scattering.upper().startswith("X")) \
         else (SFTNeutron() if isinstance(scattering, str) else scattering)
    pc.scatteringfactortable = sft

    # total + unordered pair partials in Q-space
    elems, r, n_r, dr, Gtot, Ftot, Q, dq, Fpair = compute_pair_partials_F(stru, pc)
    def F_pair(A,B): return Fpair[(A,B)] if (A,B) in Fpair else Fpair[(B,A)]

    # composition
    counts = {el: sum(1 for a in stru if a.element == el) for el in elems}
    N = float(sum(counts.values()))
    c = {el: counts[el]/N for el in elems}

    # weights:
    if use_constant_weights:
        f0 = {el: float(sft.lookup(el, 0.0)) for el in elems}
        favg0 = sum(c[el]*f0[el] for el in elems)
        # constant W_a = c_a f_a(0) / <f(0)>
        W = {el: np.full_like(Q, c[el]*f0[el]/favg0, dtype=float) for el in elems}
    else:
        # Q-dependent (can be noisy at high Q for X-rays)
        fQ = {el: np.array([sft.lookup(el, q) for q in Q]) for el in elems}
        favgQ = sum(c[el]*fQ[el] for el in elems)
        W = {el: c[el]*fQ[el]/favgQ for el in elems}

    # build shares and S_alpha(Q)
    nz = Q > 0
    S_alpha, G_alpha = {}, {}
    share = {}

    for A in elems:
        s = F_pair(A, A).copy()
        for B in elems:
            if B != A: s += 0.5 * F_pair(A, B)
        share[A] = s

        S = np.ones_like(Q)
        S[nz] += (s[nz] / Q[nz]) / W[A][nz]   # define S(0)=1
        S_alpha[A] = S

        # to real space with optional Lorch window and hard Qmax cut
        F_A = np.zeros_like(Q)
        F_A[nz] = Q[nz] * (S[nz] - 1.0)

        # enforce bandlimit and taper to kill end drop/ringing
        band = Q <= pc.qmax + 1e-9
        if use_lorch:
            F_A = F_A * lorch_window(Q, pc.qmax)
        F_A = F_A * band

        G_A_full, dr_back = fftftog(F_A, dq)
        if abs(dr_back - dr) > 1e-12:
            raise RuntimeError("dr mismatch on inverse transform.")
        G_alpha[A] = G_A_full[:n_r]   # crop to original r-grid length

    # Q-space checks
    S_tot = np.ones_like(Q); S_tot[nz] += Ftot[nz]/Q[nz]
    lhs_S = np.zeros_like(Q, dtype=float)
    for A in elems: lhs_S += W[A] * S_alpha[A]
    dev_S = np.max(np.abs(lhs_S - S_tot))

    share_sum = np.zeros_like(Q)
    for A in elems: share_sum += share[A]
    dev_F = np.max(np.abs(share_sum - Ftot))

    print("Elements:", elems)
    print("Max |Σ W_a S_a - S_total| =", dev_S)
    print("Max |Σ share_a - F_total| =", dev_F)

    return r, G_alpha, Q, S_alpha, S_tot, elems, Gtot

# ---- run example ----
if __name__ == "__main__":
    cif = "cif/BaBiO3.cif"   # <-- path to your CIF

    r, G_center, Q, S_center, S_total, elements, G_total = element_centered_dpdfs_Qspace(
        cif_path=cif,
        rmax=30.0, rstep=0.01, qmin=0.01, qmax=15.0,
        scattering="X",                # try "N" for neutrons (often cleaner O signal)
        use_constant_weights=True,     # stabilizes high-Q; keeps identity exact
        use_lorch=True                 # suppresses termination ripples / edge drop
    )

    # quick plot (optional)
    try:
        import matplotlib.pyplot as plt
        plt.figure(); plt.plot(r, G_center["Ba"], label="Ba-centered"); plt.plot(r, oBapdf, label="Ba-centered (oBapdf)"); plt.legend();
        plt.figure(); plt.plot(r, G_center["Bi"], label="Bi-centered"); plt.plot(r, oBipdf, label="Bi-centered (oBipdf)"); plt.legend();
        plt.figure(); plt.plot(r, G_center["O"], label="O-centered"); plt.plot(r, oOpdf, label="O-centered (oOpdf)"); plt.legend();
        plt.show()
    except Exception:
        pass


## In srreal

* `pdfcalc.pdf` is the reduced PDF $G(r)=\dfrac{2}{\pi}\int_0^{Q_{\max}} Q\,[S(Q)-1]\sin(Qr)\,dQ = 4\pi r\,(\rho(r)-\rho_0)$.
* With pair masks, $G_{ab}^{(\text{srreal})}(r)$ denotes the unordered, scattering-weighted pair-type contribution (e.g., Ba–O includes both Ba–O and O–Ba).
* identity (additivity of pair partials):

  $$
  G_{\text{total}}(r)=\sum_{\text{unordered }(a,b)} G_{ab}^{(\text{srreal})}(r).
  $$

* Notice element-centered (e.g. $G_{\mathrm{Ba}}=\sum_b G_{\mathrm{Ba}b}^{(\text{srreal})}$) do not equal the total when sum over elements as cross terms get double-counted.

## Differential PDFs

* Define **mixing weights** in $Q$-space

  $$
  W_\alpha(Q)=\frac{c_\alpha\,f_\alpha(Q)}{\langle f(Q)\rangle},\qquad
  \sum_\alpha W_\alpha(Q)=1,
  $$

  and decompose the measured structure function as

  $$
  S(Q)=\sum_\alpha W_\alpha(Q)\,S_\alpha(Q),\qquad S_\alpha(Q)\xrightarrow[Q\to\infty]{}1.
  $$
* The element-centered dPDF is

  $$
  G_\alpha(r)=\frac{2}{\pi}\int_0^{Q_{\max}} Q\,[S_\alpha(Q)-1]\sin(Qr)\,dQ,
  $$

* identity (element-weighted in $Q$-space):

  $$
  \sum_\alpha W_\alpha(Q)\,S_\alpha(Q)=S(Q)\quad\Rightarrow\quad
  \sum_\alpha W_\alpha(Q)\,Q[S_\alpha(Q)-1]=Q[S(Q)-1].
  $$

  In real space, the analogous identity involves the transforms of Q-space, seems not simply $\sum_\alpha w_\alpha\,G_\alpha(r)=G(r)$ with a constant $w_\alpha$.


# Exact r-space relationships

Because $S_\alpha(Q)=\sum_\beta W_\beta(Q)S_{\alpha\beta}(Q)$ and the transform is linear, the exact r-space relation should be an integral kernel (sine-transform convolution):

$$
G_\alpha(r)=\sum_{\beta}\int_0^\infty 
K_\beta(r,r')\,G_{\alpha\beta}(r')\,dr'
$$

with the kernel

$$
K_\beta(r,r') \equiv \frac{2}{\pi}\int_0^{Q_{\max}} W_\beta(Q)\,\sin(Qr)\,\sin(Qr')\,dQ.
$$

$$
S(Q)=\sum_\alpha W_\alpha(Q)\,S_\alpha(Q)
\;\Longrightarrow\;
G(r)=\sum_\alpha\int_0^\infty 
\underbrace{\Big[\frac{2}{\pi}\!\int_0^{Q_{\max}}\! W_\alpha(Q)\sin(Qr)\sin(Qr')\,dQ\Big]}_{\tilde K_\alpha(r,r')}
\,G_\alpha(r')\,dr'.
$$

not a simple constant-weighted sum in r-space unless the weights are Q-independent.


# Constant-weight (Morningstar–Warren) approximation

If take $f_\gamma(Q)\approx f_\gamma(0)$ so that $W_\beta(Q)\equiv W_\beta$ is **constant**, the kernel collapses to a delta (up to finite-$Q_{\max}$ resolution), and the r-space relations simplify to ordinary sums:

  $$
  G_\alpha(r)\;\approx\;\sum_{\beta} W_\beta\,G_{\alpha\beta}(r)
  $$

  $$
  G(r)\;\approx\;\sum_{\alpha} W_\alpha\,G_\alpha(r)
  $$

  $$
  G(r)\;\approx\;\sum_{\alpha\beta}
  \frac{c_\alpha c_\beta f_\alpha(0)f_\beta(0)}{\langle f(0)\rangle^2}\,G_{\alpha\beta}(r)
  $$